In [1]:
import json
import pandas as pd 
import gzip


In [2]:
with gzip.open('../ga4-raw-data/events-json/analytics_291746817/2024/10/events_20241001.json', 'rt', encoding='utf-8') as f:
    data = json.load(f)


In [3]:
print(json.dumps(data, indent=4, ensure_ascii=False))


[
    {
        "event_date": "20241001",
        "event_timestamp": 1727797576466109,
        "event_name": "user_session_info",
        "event_params": [
            {
                "key": "ga_session_number",
                "value": {
                    "string_value": null,
                    "int_value": 1,
                    "float_value": null,
                    "double_value": null
                }
            },
            {
                "key": "session_engaged",
                "value": {
                    "string_value": "0",
                    "int_value": null,
                    "float_value": null,
                    "double_value": null
                }
            },
            {
                "key": "page_referrer",
                "value": {
                    "string_value": "https://www.bing.com/",
                    "int_value": null,
                    "float_value": null,
                    "double_value": null
                }
       

In [4]:
data_df = pd.json_normalize(data,sep='_')
data_df

,event_date,event_timestamp,event_name,event_params,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,...,collected_traffic_source_manual_medium,collected_traffic_source_manual_term,collected_traffic_source_manual_content,collected_traffic_source_manual_source_platform,collected_traffic_source_manual_creative_format,collected_traffic_source_manual_marketing_tactic,collected_traffic_source_gclid,collected_traffic_source_dclid,collected_traffic_source_srsltid,device_web_info
0,20241001,1727797576466109,user_session_info,"[{'key': 'ga_session_number', 'value': {'strin...",None,None,952762045,None,None,1855006594.1727797571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,20241001,1727797576466109,scroll,"[{'key': 'batch_ordering_id', 'value': {'strin...",None,None,952762045,None,None,1855006594.1727797571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20241001,1727797576466109,scroll,"[{'key': 'session_engaged', 'value': {'string_...",None,None,952762045,None,None,1855006594.1727797571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,20241001,1727797576466109,scroll,"[{'key': 'session_engaged', 'value': {'string_...",None,None,952762045,None,None,1855006594.1727797571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,20241001,1727797698906576,user_engagement,"[{'key': 'session_engaged', 'value': {'string_...",None,None,1075202512,None,None,1855006594.1727797571,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,20241001,1727796290320198,session_start,"[{'key': 'page_location', 'value': {'string_va...",None,None,-333383866,None,None,1065764961.1724430926,...,GMBandOtherOffSiteDirectories,(not provided),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
890,20241001,1727796290320198,page_view,"[{'key': 'page_location', 'value': {'string_va...",None,None,-333383866,None,None,1065764961.1724430926,...,GMBandOtherOffSiteDirectories,(not provided),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
891,20241001,1727796345794529,page_view,"[{'key': 'batch_ordering_id', 'value': {'strin...",None,None,-277909535,None,None,1065764961.1724430926,...,organic,(not provided),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
892,20241001,1727796345794529,user_session_info,"[{'key': 'session_engaged', 'value': {'string_...",None,None,-277909535,None,None,1065764961.1724430926,...,organic,(not provided),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
params_data =[
    {item['key']:next(v for v in item['value'].values() if v is not None)
     for item in params_list}
    for params_list in data_df['event_params']
]

In [6]:
param_df = pd.DataFrame(params_data).add_prefix('ep_')

In [7]:
user_props_data =[
    {
        prop['key']:next(v for k, v in prop['value'].items()if v is not None and k!='set_timestamp_micros')
        for prop in props_list
    }
    for props_list in data_df['user_properties']
]

In [8]:
user_props_df = pd.DataFrame(user_props_data).add_prefix('user_prop_')


In [9]:
event_date_dt = pd.to_datetime(data_df["event_date"], format="%Y%m%d")
data_df['year']= event_date_dt.dt.year
data_df['month'] = event_date_dt.dt.month

In [10]:
final_df = pd.concat([data_df, param_df, user_props_df], axis=1)
final_df.drop(columns=['user_properties', 'event_params'], inplace=True)

In [11]:
final_df

,event_date,event_timestamp,event_name,event_previous_timestamp,event_value_in_usd,event_bundle_sequence_id,event_server_timestamp_offset,user_id,user_pseudo_id,user_first_touch_timestamp,...,ep_tracking_number,ep_company_name,ep_browser_or_device,ep_medium,ep_referrer,ep_call_id,ep_campaign,ep_term,user_prop_user_session_id,user_prop_user_client_id
0,20241001,1727797576466109,user_session_info,None,None,952762045,None,None,1855006594.1727797571,1.727798e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,_1727797571,_1855006594.1727797571
1,20241001,1727797576466109,scroll,None,None,952762045,None,None,1855006594.1727797571,1.727798e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,_1727797571,_1855006594.1727797571
2,20241001,1727797576466109,scroll,None,None,952762045,None,None,1855006594.1727797571,1.727798e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,_1727797571,_1855006594.1727797571
3,20241001,1727797576466109,scroll,None,None,952762045,None,None,1855006594.1727797571,1.727798e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,_1727797571,_1855006594.1727797571
4,20241001,1727797698906576,user_engagement,None,None,1075202512,None,None,1855006594.1727797571,1.727798e+15,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
889,20241001,1727796290320198,session_start,None,None,-333383866,None,None,1065764961.1724430926,1.724431e+15,...,NaN,NaN,NaN,GMBandOtherOffSiteDirectories,NaN,NaN,(organic),(not provided),NaN,NaN
890,20241001,1727796290320198,page_view,None,None,-333383866,None,None,1065764961.1724430926,1.724431e+15,...,NaN,NaN,NaN,GMBandOtherOffSiteDirectories,NaN,NaN,(organic),(not provided),NaN,NaN
891,20241001,1727796345794529,page_view,None,None,-277909535,None,None,1065764961.1724430926,1.724431e+15,...,NaN,NaN,NaN,organic,NaN,NaN,(organic),(not provided),NaN,NaN
892,20241001,1727796345794529,user_session_info,None,None,-277909535,None,None,1065764961.1724430926,1.724431e+15,...,NaN,NaN,NaN,organic,NaN,NaN,(organic),(not provided),_1727796290,_1065764961.1724430926


In [12]:
for col in final_df.columns:
    if final_df[col].dtype == 'object':
        final_df[col] = final_df[col].astype(str)

In [13]:
final_df.to_parquet('C:\\Users\\aalig\\Downloads\\ga.parquet', index=False)